In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [3]:
!ls

bert_tokenizer	 FDA_therapy_biomarker_extraction.ipynb  therapy_links
biomarker_nlp	 negCue					 Untitled0.ipynb
biomarker_nlp_1  negScope


In [ ]:
import sys
import transformers.utils

# 1. Manually route the missing cached_path to the modern hub_utils
try:
    from transformers.utils import hub_utils
    # Create the missing reference in the expected legacy location
    sys.modules['transformers.file_utils'] = transformers.utils
    transformers.file_utils.cached_path = hub_utils.cached_file
except ImportError:
    # Fallback for very specific versions
    import transformers.file_utils
    transformers.file_utils.cached_path = lambda x: x

print(" Transformers legacy patch applied.")

In [6]:
!ls

bert_tokenizer	 FDA_therapy_biomarker_extraction.ipynb  therapy_links
biomarker_nlp	 negCue					 Untitled0.ipynb
biomarker_nlp_1  negScope


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [7]:
with open('therapy_links','r') as file:
  therapies = file.read().split(',')

In [8]:
!ls

bert_tokenizer	 FDA_therapy_biomarker_extraction.ipynb  therapy_links
biomarker_nlp	 negCue					 Untitled0.ipynb
biomarker_nlp_1  negScope


In [10]:
!pip install scispacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 42.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 re

In [11]:
from biomarker_nlp import biomarker_extraction

#### HTML Parsing using Biomarker Extraction to extract disease name,Drug Label, Raw text of Drug Information

In [12]:
import pandas as pd
from tqdm import tqdm

skeleton_data = []
for url in tqdm(therapies, desc="Pass 1: Metadata"):
    full_url = f"https://www.cancer.gov{url}" if url.startswith('/') else url
    try:
        name = biomarker_extraction.targeted_therapy_name(full_url)[0]
        diseases = biomarker_extraction.therapy_disease(full_url)
        dm_url = biomarker_extraction.drug_search_url(full_url)[0]

        brand = biomarker_extraction.drug_brand_label(dm_url)
        ndc = biomarker_extraction.ndc_code(dm_url)

        for dis in diseases:
            content = biomarker_extraction.disease_content(dm_url, dis)
            skeleton_data.append({
                "therapy_label": name,
                "disease_label": dis,
                "drug_label": brand,
                "NDCCode_label": ndc,
                "raw_text": content
            })
    except: continue

df = pd.DataFrame(skeleton_data)
df.to_csv("checkpoint_1.csv", index=False)

Pass 1: Metadata: 100%|██████████| 389/389 [32:50<00:00,  5.06s/it]


### Stage 2 Idetifying geneProtein_Label from Raw Text using the Spacy Model

In [2]:
import pandas as pd

In [1]:
import os
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [3]:
df = pd.read_csv("checkpoint_1.csv")


In [4]:
df.head(5)

,therapy_label,disease_label,drug_label,NDCCode_label,raw_text
0,Abemaciclib,Breast cancer,VERZENIO- abemaciclib tablet,"0002-4483-54, 0002-4483-62, 0002-4815-54, 0002...",\nVERZENIO® (abemaciclib) is indicated:\n\n\n\...
1,Abiraterone Acetate,Prostate cancer,ZYTIGA- abiraterone acetate tablet,"57894-150-12, 57894-150-25, 57894-195-06, 5789...",NaN
2,Acalabrutinib Maleate Monohydrate,Chronic lymphocytic leukemia,"CALQUENCE- acalabrutinib capsule, gelatin coated","0310-0512-28, 0310-0512-60, 0310-0512-95",\nCALQUENCE is indicated for the treatment of ...
3,Acalabrutinib Maleate Monohydrate,small lymphocytic lymphoma,"CALQUENCE- acalabrutinib capsule, gelatin coated","0310-0512-28, 0310-0512-60, 0310-0512-95",\nCALQUENCE is indicated for the treatment of ...
4,Acalabrutinib Maleate Monohydrate,Mantle cell lymphoma,"CALQUENCE- acalabrutinib capsule, gelatin coated","0310-0512-28, 0310-0512-60, 0310-0512-95",\nCALQUENCE in combination with bendamustine a...


### Installing the Pre-Trained Model

In [6]:
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 139.0 MB/s eta 0:00:00
  Created wheel for en_ner_bionlp13cg_md: filename=en_ner_bionlp13cg_md-0.5.4-py3-none-any.whl size=119814670 sha256=b1b98715ed976b4a9a8dcc76ed331c5c3dbd0c55527a55f54c19d4af719a3862
  Stored in directory: /root/.cache/pip/wheels/95/ca/35/7a25decd9a4965ccae588698b6d96f92e1c6816d805f509590
Successfully built en_ner_bionlp13cg_md
  Attempting uninstall: blis
    Found existing installation: blis 1.3.3
    Uninstalling blis-1.3.3:
      Successfully uninstalled blis-1.3.3
  Attempting uninstall: thinc
    Found existing installation: thinc 8.3.10
    Uninst

In [ ]:
import spacy
import torch
import gc

df = pd.read_csv("checkpoint_1.csv")

spacy.require_gpu()
nlp = spacy.load("en_ner_bionlp13cg_md", disable=["tagger", "parser", "lemmatizer"])

def extract_genes(text):
    if pd.isna(text) or not isinstance(text, str): return "*"
    doc = nlp(text)
    found = [ent.text for ent in doc.ents if ent.label_ in ["GENE_OR_GENE_PRODUCT", "PROTEIN"]]
    return ", ".join(list(set(found))) if found else "*"

df["geneProtein_label"] = df["raw_text"].apply(extract_genes)
df.to_csv("biomarker_output.csv", index=False)

del nlp
gc.collect()
torch.cuda.empty_cache()

#### STAGE 3 Identify Binary Classification Columns (Metastatic,AcceleratedApproval,Firsline)

In [1]:
import os

In [3]:
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [4]:
import sys
import huggingface_hub

import transformers.utils.hub as hub

try:
    from huggingface_hub import hf_hub_download
    hub.cached_path = hf_hub_download
except ImportError:
    !pip install huggingface_hub
    from huggingface_hub import hf_hub_download
    hub.cached_path = hf_hub_download

print("Patched transformers.utils.hub.cached_path")

✅ Patched transformers.utils.hub.cached_path


In [12]:
!pip install knockknock

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.1/180.1 kB 29.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [6]:
import spacy

In [15]:
nlp = spacy.load("en_ner_bionlp13cg_md", disable=["tagger", "lemmatizer"])


In [11]:
import torch
import __main__
import inspect
import gc
import torch.nn.functional as F
import transformers.models.xlnet.modeling_xlnet as xlnet_mod
from transformers import XLNetForTokenClassification, XLNetModel
from biomarker_nlp.negation_cue_scope import CueModel, ScopeModel


__main__.gelu = F.gelu
__main__.relu = F.relu

for name, obj in inspect.getmembers(xlnet_mod):
    if inspect.isclass(obj):
        setattr(__main__, name, obj)

__main__.CueModel = CueModel
__main__.ScopeModel = ScopeModel
__main__.XLNetForTokenClassification = XLNetForTokenClassification
__main__.XLNetModel = XLNetModel

In [12]:
modelCue = torch.load('/content/drive/MyDrive/Biomarker_Project/negCue', weights_only=False)

In [13]:
modelScope = torch.load('/content/drive/MyDrive/Biomarker_Project/negScope', weights_only=False)

In [9]:
import pandas as pd
df = pd.read_csv('biomarker_output.csv')

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from biomarker_nlp import negation_cue_scope

tqdm.pandas(desc="Processing Stage 3")

def process_stage_3_final(row, nlp, modelCue, modelScope):
    text = str(row.get('raw_text', ''))
    disease = str(row.get('disease_label', '')).lower()

    if len(text) < 10:
        return "", "", "", ""

    doc = nlp(text)

    metastatic = ""
    first_line = ""
    accel_app = ""

    for token in doc:
        t_low = token.text.lower()

        if t_low in ["metastatic", "advanced", "recurrent", "stage iv"]:
            if token.dep_ in ["amod", "compound", "conj"]:
                metastatic = "Yes"
            elif any(d_word in [t.text.lower() for t in token.head.subtree] for d_word in disease.split()):
                metastatic = "Yes"

        if "first-line" in t_low or "untreated" in t_low:
            first_line = "Yes"

    if "accelerated approval" in text.lower():
        accel_app = "Yes"

    neg_scope_text = ""
    try:
        with torch.inference_mode():
            if negation_cue_scope.negation_detect(text, modelCue):
                scope_tokens = negation_cue_scope.negation_scope(text, modelCue, modelScope)
                neg_scope_text = " ".join(scope_tokens).lower()

                if metastatic == "Yes" and "metastatic" in neg_scope_text:
                    metastatic = ""
                if first_line == "Yes" and "first-line" in neg_scope_text:
                    first_line = ""
    except:
        pass

    return metastatic, first_line, accel_app, neg_scope_text

print("🚀 Initializing Stage 3: Deep Learning Analysis...")

results = df.progress_apply(
    lambda row: process_stage_3_final(row, nlp, modelCue, modelScope),
    axis=1
)

df[['metastatic', 'firstLine', 'acceleratedApproval', 'negGeProDr_label']] = pd.DataFrame(results.tolist(), index=df.index)

print("\n✅ Stage 3 Complete!")

In [22]:
df.to_csv('biomarker_report_final.csv', index=False)

### USES IF-ELSE

#### Restart Session

In [23]:
import pandas as pd

In [24]:
import os
os.chdir('/content/drive/MyDrive/Biomarker_Project')

In [25]:
import pandas as pd
import numpy as np
import gc

df = pd.read_csv("biomarker_report_final.csv")
def stage_4_clean_robust(df):
    nlp_columns = [
        "geneProtein_label", "combinationDrug_label", "negGeProDr_label",
        "firstLine", "acceleratedApproval", "acceleratedApprovalRate", "metastatic"
    ]
    for col in nlp_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).replace(['nan', 'None', 'error', 'NoneType'], '')

            df[col] = df[col].str.strip()
            df[col] = df[col].apply(lambda x: "" if x.lower() == "error" else x)
    df = df.drop_duplicates().reset_index(drop=True)

    final_order = [
        "geneProtein_label", "therapy_label", "disease_label", "drug_label",
        "NDCCode_label", "combinationDrug_label", "negGeProDr_label",
        "firstLine", "acceleratedApproval", "acceleratedApprovalRate", "metastatic"
    ]
    existing_cols = [col for col in final_order if col in df.columns]
    df_final = df[existing_cols]
    output_filename = "Final_Biomarker_Report.csv"
    df_final.to_csv(output_filename, index=True, index_label="ID")
    return output_filename, df_final

final_path, df_result = stage_4_clean_robust(df)

del df
gc.collect()

print(f" Success! {len(df_result)} rows cleaned.")

 Success! 708 rows cleaned.


In [ ]:
import pandas as pd
import gc

try:
    df = pd.read_csv("biomarker_report_final.csv")
except:
    df = pd.read_csv("biomarker_output.csv")

def stage_4_finalize(df):
    print("🧹 Finalizing report: Dropping geneProtein_label...")

    if 'geneProtein_label' in df.columns:
        df.drop(columns=['geneProtein_label'], inplace=True)

    if 'raw_text' in df.columns:
        df.drop(columns=['raw_text'], inplace=True)

    df = df.astype(str).replace(['nan', 'None', 'error', 'NoneType', '*'], '')
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    final_order = [
        "therapy_label",
        "disease_label",
        "drug_label",
        "NDCCode_label",
        "combinationDrug_label",
        "negGeProDr_label",
        "firstLine",
        "acceleratedApproval",
        "acceleratedApprovalRate",
        "metastatic"
    ]

    existing_cols = [col for col in final_order if col in df.columns]
    df_final = df[existing_cols]

    df_final = df_final.drop_duplicates().reset_index(drop=True)

    output_name = "Final_Clinical_Biomarker_Report.csv"
    df_final.to_csv(output_name, index=True, index_label="ID")

    return output_name

#
final_csv = stage_4_finalize(df)

del df
gc.collect()